# 09 — Persistent Matmul

**Ladder:** Classic one program / tile (tutorial 03) → persistent schedule (few workers, many tiles) → Triton persistent kernel (`tl.num_programs` / work queue).

**Study tip.** Persistence changes **who runs which tile**, not $C=AB$. Learn the static stride $p,p+P,\dots$ first; then ask when an atomic work queue helps.

Official: [Persistent Matmul](https://triton-lang.org/main/getting-started/tutorials/09-persistent-matmul.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: classic one-tile programs vs persistent workers](docs/images/benchmark-persistent-matmul.png)

*Theory board — huge 1:1 tile grid vs few resident programs that steal tiles from a work queue.*

---

# Part I — Mathematics

## 0. Two schedules for the same tiles

Let the output tile grid have

$$
T=G_MG_N,
\qquad
G_M=\left\lceil\frac{M}{B_M}\right\rceil,
\qquad
G_N=\left\lceil\frac{N}{B_N}\right\rceil
\tag{0.1}
$$

tiles. The **math** per tile is the blocked product from notebook 03:

$$
C_{\mathrm{tile}}=\sum_{t=0}^{G_K-1}A_tB_t,
\qquad
G_K=\left\lceil\frac{K}{B_K}\right\rceil,
\tag{0.2}
$$

with an FP32 accumulator.

| Schedule | Grid size | Tile assignment |
|---|---|---|
| **Classic (naive launch)** | $T$ | program $p$ does tile $p$ once |
| **Persistent (static)** | $P\ll T$ | program $p$ does tiles $p,p+P,p+2P,\dots$ |
| **Work queue** | $P\ll T$ | `tile_id ← atomicAdd(counter)` ("tile stealing") |

**Lemma 1 (stride coverage).** For a persistent launch with $P$ programs,

$$
\bigcup_{p=0}^{P-1}\{p+kP\mid k\ge0,\ p+kP<T\}=\{0,\dots,T-1\}.
\tag{0.3}
$$

*Proof.* Every $t<T$ has unique quotient/remainder $t=p+kP$ with
$p=t\bmod P\in[0,P)$ and $k=\lfloor t/P\rfloor\ge0$. $\square$

**Corollary 2.** The stride schedule computes every tile exactly once, hence the
same $C$ as the classic launch.

*Proof.* Lemma 1 plus the tile bijection of notebook 03, §2. $\square$

### 0.1 Work per program and imbalance

With the stride schedule, program $p$ processes

$$
n_p=\left\lceil\frac{T-p}{P}\right\rceil
=
\left\lfloor\frac{T+P-1-p}{P}\right\rfloor
\tag{0.4}
$$

tiles. Hence

$$
\left\lfloor\frac{T}{P}\right\rfloor\le n_p\le\left\lceil\frac{T}{P}\right\rceil,
\qquad
\max_p n_p=\left\lceil\frac{T}{P}\right\rceil.
\tag{0.5}
$$

If every tile costs $t_{\mathrm{tile}}$, the makespan is

$$
T_{\mathrm{persistent}}
=
\left\lceil\frac{T}{P}\right\rceil t_{\mathrm{tile}},
\tag{0.6}
$$

and the **scheduling efficiency** is

$$
\eta
=
\frac{T\,t_{\mathrm{tile}}}{P\cdot\lceil T/P\rceil\,t_{\mathrm{tile}}}
=
\frac{T}{P\lceil T/P\rceil}
\;\in\;\left(\frac{T}{T+P},\,1\right],
\tag{0.7}
$$

i.e. $\eta=1$ whenever $P\mid T$, and $\eta\approx1-\frac{P}{T}$ otherwise.
For $T\gg P$ the imbalance is negligible.

### 0.2 Wave quantization and the tail effect

A GPU can host at most

$$
P_{\max}=\text{SMs}\times\text{blocks per SM}
\tag{0.8}
$$

concurrent programs. The classic launch schedules $T$ blocks in waves of
$P_{\max}$:

$$
W_{\mathrm{classic}}=\left\lceil\frac{T}{P_{\max}}\right\rceil.
\tag{0.9}
$$

The last wave may contain as few as one block, so the machine idles for most of
it. The **tail fraction** is

$$
\text{waste}
=
\frac{W_{\mathrm{classic}}P_{\max}-T}{W_{\mathrm{classic}}P_{\max}}
\;\le\;
1-\frac{1}{W_{\mathrm{classic}}}.
\tag{0.10}
$$

A persistent launch chooses $P\le P_{\max}$ and performs the same total work,
but every program stays resident and pulls more tiles; there is no per-tile CTA
launch and the tail is one **tile**, not one wave. This is the main performance
argument for persistence on irregular or many-tile problems.

**Corollary 3.** Classic and persistent execute the same number of tile-units;
persistence can only reduce launch/tail overhead, never the arithmetic.

### 0.3 Work queue

A static stride schedule assumes all tiles cost the same. If they do not (e.g.
different $K$ per group in a Group GEMM, or cache-dependent variation), a
work-stealing queue balances the load:

$$
\text{next}=\operatorname{atomicAdd}(\mathrm{counter},1).
\tag{0.11}
$$

**Lemma 4 (safety).** The atomic counter returns each integer in $[T)$ exactly
once, so each tile is computed exactly once.

*Proof.* `atomicAdd` is a linearizable read-modify-write; the sequence of return
values is a permutation of $0,1,\dots,T-1$. $\square$

**Cost.** One atomic per tile, $T$ atomics total. This is negligible compared
with $O(T\,B_MB_NK)$ FLOPs, and much cheaper than per-tile CTA launches in the
classic schedule. The price is non-determinism: which program computes which
tile can change between runs (the **result** is unaffected).

### 0.4 Choosing $P$

- $P$ too small → not enough parallelism, memory latency exposed.
- $P=P_{\max}$ → full occupancy, but every program must fit in registers/SMEM.
- $P$ slightly below $P_{\max}$ → sometimes better cache behavior (fewer
  competing working sets).
- $P>T$ → wasted programs; no tile to do.

The implementation defaults to

$$
P=\min(T,\ \text{small multiple of SMs}),
\tag{0.12}
$$

which is a heuristic: enough programs to fill the machine, few enough to stay
resident.

### 0.5 L2 locality

The stride schedule makes successive tiles of a program far apart:

$$
p,\ p+P,\ p+2P,\dots
\tag{0.13}
$$

This is good for **global** coverage but poor for **per-program** locality. Two
refinements:

1. **Contiguous chunks.** Program $p$ takes tiles
   $[p\lceil T/P\rceil,(p+1)\lceil T/P\rceil)$; better per-program reuse,
   slightly worse balance.
2. **Grouped swizzle.** Apply the tutorial-03 grouped order to the tile id
   before assigning it; consecutive tiles then share $A$ or $B$ panels.

Persistence and swizzling are orthogonal: persistence decides **how many**
workers, swizzling decides **which** tiles they touch.

### 0.6 Teaching sizes

$M=64$, $N=48$, $B_M=32$, $B_N=16$ → $G_M=2$, $G_N=3$, $T=6$. Take $P=3$:
program $p$ does tiles $\{p,p+3\}$, i.e. $n_p=2$ for all $p$, $\eta=1$.

### Theory board

![Whiteboard: classic one-tile programs vs persistent workers](docs/images/whiteboard-theory.jpg)

*Theory board — huge 1:1 tile grid vs few resident programs that steal tiles from a work queue.*

# Part II — Proof

**Lemma (stride coverage).** For $P\ge1$ and $T\ge1$,

$$
\bigcup_{p=0}^{P-1}\{p+kP\mid k\ge0,\ p+kP<T\}=\{0,\dots,T-1\}.
$$

*Proof.* Every $t<T$ has unique quotient/remainder $t=p+kP$ with
$p=t\bmod P\in[0,P)$ and $k=\lfloor t/P\rfloor\ge0$. $\square$

**Theorem (persistent correctness).** A persistent launch with $P$ programs,
each executing the tile body for tile ids $p,p+P,p+2P,\dots$, computes the same
$C=AB$ as the classic launch.

*Proof.* Lemma gives exactly-once tile coverage; the tile body is the blocked
product of notebook 03. $\square$

**Corollary (efficiency).** With $n_p=\lceil(T-p)/P\rceil$ tiles per program,
$\lfloor T/P\rfloor\le n_p\le\lceil T/P\rceil$ and the scheduling efficiency is

$$
\eta=\frac{T}{P\lceil T/P\rceil}\in\left(\frac{T}{T+P},1\right].
$$

*Proof.* The maximum tile count is $\lceil T/P\rceil$; the bound follows from
$\lceil T/P\rceil<T/P+1$. $\square$

# Part III — Step-by-step proof

1. **Build the tile grid.** $G_M=\lceil M/B_M\rceil$,
   $G_N=\lceil N/B_N\rceil$, $T=G_MG_N$.
2. **Choose the grid size.** $P\le P_{\max}$ resident programs; classic uses
   $P=T$, persistent uses $P\ll T$.
3. **Assign tiles.** Program $p$ processes $p,p+P,p+2P,\dots<T$.
4. **Map tile to indices.** $p_m=\lfloor t/G_N\rfloor$,
   $p_n=t\bmod G_N$.
5. **Run the tile body.** Load $K$-panels, `tl.dot`, accumulate in FP32.
6. **Store.** Write the tile to $C$.
7. **Repeat.** The loop bound guarantees coverage and termination.
8. **Tail analysis.** Classic waits for a full wave; persistent waits for at
   most one tile.
9. **Work queue variant.** Replace step 3 by
   $t=\operatorname{atomicAdd}(\mathrm{counter},1)$; the counter returns each
   tile id exactly once.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


try:
    _HERE = Path(__file__).resolve().parent
except NameError:  # Jupyter notebook: __file__ is undefined
    _HERE = Path.cwd()
_REPO = _HERE
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "03_matrix_multiplication" / "tutorial.py").exists():
        _REPO = _candidate
        break
else:
    raise FileNotFoundError(
        "cannot locate 03_matrix_multiplication from " + str(_HERE)
    )
_matmul_path = _REPO / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


### Method 01 — Naive (Python tiled reference)

The reference loops over tiles in Python and computes each tile with
`torch.matmul`. The loop is written with an explicit `num_programs`; setting it equal to
the number of tiles degenerates to the classic schedule, while a smaller
value shows the persistence idea in pure Python.


In [ ]:
def persistent_matmul_pseudo(A, B, BLOCK_M, BLOCK_N, num_programs):
    """
    Persistent idea in Python:
    (See markdown cells above for the math.)
    """
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    assert M % BLOCK_M == 0 and N % BLOCK_N == 0

    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)
    grid_m = M // BLOCK_M
    grid_n = N // BLOCK_N
    num_tiles = grid_m * grid_n

    for pid in range(num_programs):
        for tile_id in range(pid, num_tiles, num_programs):
            pid_m = tile_id // grid_n
            pid_n = tile_id % grid_n
            m0 = pid_m * BLOCK_M
            n0 = pid_n * BLOCK_N
            C[m0 : m0 + BLOCK_M, n0 : n0 + BLOCK_N] = (
                A[m0 : m0 + BLOCK_M, :] @ B[:, n0 : n0 + BLOCK_N]
            )
    return C


### Method 02 — Kernel-based (tutorial-03 GEMM)

Each tile is computed by the tutorial-03 Triton GEMM (`matmul`), launched as a
regular grid. It is the kernel-based baseline; the persistent schedule follows.


In [ ]:
def naive_blocked_matmul(A, B):
    """
    Classic blocked launch: grid ≈ (#tiles_M) * (#tiles_N).
    Each program owns exactly one output tile, then exits.

    We reuse tutorial 03's `matmul` (fused grouped GEMM) as that baseline.
    """
    return matmul(A, B)


### Method 03 — Fused kernel-based (persistent)

A small resident grid of $P$ programs strides over the tile ids
$p,p+P,p+2P,\dots$. Each program stays alive and processes many tiles, so the
per-tile launch cost and the wave-sized tail are removed. The tile body is
identical to Method 02.


In [ ]:
@triton.jit
def persistent_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """Few programs; each strides across output tiles with tl.num_programs."""
    pid = tl.program_id(axis=0)
    num_pid = tl.num_programs(axis=0)

    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_tiles = num_pid_m * num_pid_n

    for tile_id in range(pid, num_tiles, num_pid):
        pid_m = tile_id // num_pid_n
        pid_n = tile_id % num_pid_n

        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        offs_k = tl.arange(0, BLOCK_K)

        a_ptrs = A_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
        b_ptrs = B_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        for k in range(0, tl.cdiv(K, BLOCK_K)):
            k_remaining = K - k * BLOCK_K
            a = tl.load(
                a_ptrs,
                mask=(offs_m[:, None] < M) & (offs_k[None, :] < k_remaining),
                other=0.0,
            )
            b = tl.load(
                b_ptrs,
                mask=(offs_k[:, None] < k_remaining) & (offs_n[None, :] < N),
                other=0.0,
            )
            acc += tl.dot(a, b)
            a_ptrs += BLOCK_K * stride_ak
            b_ptrs += BLOCK_K * stride_bk

        c = acc.to(C_ptr.dtype.element_ty)
        c_ptrs = C_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
        tl.store(c_ptrs, c, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


def persistent_matmul(a, b, num_programs=None, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32):
    """Launch a small persistent grid; each program loops over many tiles."""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    num_tiles = triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N)
    if num_programs is None:
        try:
            sms = torch.cuda.get_device_properties(a.device).multi_processor_count
        except Exception:
            sms = 8  # CPU / interpreter fallback
        num_programs = min(num_tiles, sms)
    persistent_matmul_kernel[(num_programs,)](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return c


### Pitfalls

- Persistent grid size ≈ #SMs (or a small multiple), not #tiles.
- Need a correct work counter / tile-id mapping.
- Over-persistence without enough tiles can hurt latency.
- Forgetting the final `atomicAdd` bound can read past the tile count.

### Self-check

1. Prove Lemma 1 and the efficiency formula (0.7).
2. Contrast classic grid vs persistent worker loop.
3. Who decides the next tile — host or device counter?
4. Compute the tail waste for $T=100$, $P_{\max}=32$.
5. When is persistence most helpful?


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(64, 32)
    b = torch.randn(32, 48)
    ref = a @ b
    out = persistent_matmul_pseudo(a, b, BLOCK_M=32, BLOCK_N=16, num_programs=3)
    assert torch.allclose(out, ref, atol=1e-5)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    a = torch.randn(256, 128, device=DEVICE, dtype=torch.float16)
    b = torch.randn(128, 256, device=DEVICE, dtype=torch.float16)
    ref = torch.matmul(a, b)

    baseline = naive_blocked_matmul(a, b)
    assert torch.allclose(baseline.float(), ref.float(), atol=1e-2, rtol=1e-2)

    out = persistent_matmul(a, b, num_programs=4, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)
    assert torch.allclose(out.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("persistent matmul OK")
